## **Claude SDK Tutorial**

https://anthropic.skilljar.com/claude-with-the-anthropic-api

---

In [1]:
# Install dependencies
%pip install anthropic python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# Load env variables
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
# Create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-sonnet-4-0"

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

# Make a request
def chat(messages: list[object], system_prompt=None, temperature=1.0, stop_sequences=[]) -> str:

    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences
    }

    if system_prompt:
        params["system"] = system_prompt

    message = client.messages.create(**params) # unpack dict into keyword args
    return message.content[0].text

In [5]:
# Start with an empty message list
messages = []

# Add the initial user question
add_user_message(messages, "What is quantum computing")

# Get Claude's response
answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, answer)

# Add a follow-up question
add_user_message(messages, "Write another sentence")

# Get the follow-up response with full context
final_answer = chat(messages)

# Add Claude's response to the conversation history
add_assistant_message(messages, final_answer)

# Print the full message history
# messages

# Print final answer
final_answer

'Quantum computing represents one of the most promising yet challenging frontiers in technology, where the bizarre laws of quantum physics could eventually solve problems that would take classical computers longer than the age of the universe to complete.'

In [6]:
# Implement looping chatbot
messages = []

while True:
    user_input = input("> ")
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print(">", answer)
    print("---")

> Hello! How are you doing today? Is there anything I can help you with?
---
> I'm doing well, thank you for asking! I'm here and ready to help with whatever you might need - whether that's answering questions, having a conversation, helping with tasks, or just chatting. What's on your mind today?
---


BadRequestError: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.4: user messages must have non-empty content'}, 'request_id': 'req_011CaViAef4yEH3a8hJLNMuj'}

In [7]:
messages = []

add_user_message(messages, "write a python function that checks string for duplicate characters")

answer = chat(messages, system_prompt="You are a python engineer who writes very concise code")

answer

"```python\ndef has_duplicates(s):\n    return len(s) != len(set(s))\n```\n\nThis function converts the string to a set (which removes duplicates) and compares lengths. If they're different, duplicates exist."

---

## **Understanding Temperature**

![temperature](temperature.png)

In [8]:
messages = []

add_user_message(messages, "tell a joke")

answer = chat(messages, temperature=1.0)

answer

"Why don't scientists trust atoms?\n\nBecause they make up everything!"

---

## **Streaming**

In [9]:
# Manaul streaming
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

stream = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    stream=True
)

for event in stream:
    print(event)

RawMessageStartEvent(message=Message(id='msg_01HDjhHa2axdJPs5LBiX9CaW', container=None, content=[], model='claude-sonnet-4-20250514', role='assistant', stop_reason=None, stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=15, output_tokens=6, server_tool_use=None, service_tier='standard'), stop_details=None), type='message_start')
RawContentBlockStartEvent(content_block=TextBlock(citations=None, text='', type='text'), index=0, type='content_block_start')
RawContentBlockDeltaEvent(delta=TextDelta(text='A fake database is a sim', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta=TextDelta(text='ulated or mock data storage system that contains artificially generated, placeholder', type='text_delta'), index=0, type='content_block_delta')
RawContentBlockDeltaEvent(delta

In [10]:
# Built-in streaming capability
messages = []

add_user_message(messages, "Write one sentence description of a fake database")

with client.messages.stream(
    model=model,
    max_tokens=1000,
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="")
        # pass

# Collect all events e.g., For record keeping in DB
# stream.get_final_message()

A fake database is a simulated or mock data repository that contains fabricated records designed to mimic real data structures and relationships for testing, development, or demonstration purposes without exposing actual sensitive information.

---

## **Structured Data Using Prefill Assistant Message & Stop-Sequences**

In [11]:
messages = []

add_user_message(messages, "Generate very short event bridge rule as json")

# Prefill assistant message
add_assistant_message(messages, "```json") # writes out response after the text containing "json"

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\n{\n  "Name": "my-rule",\n  "EventPattern": {\n    "source": ["my.app"],\n    "detail-type": ["Order Placed"]\n  },\n  "Targets": [\n    {\n      "Id": "1",\n      "Arn": "arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder"\n    }\n  ]\n}\n'

In [12]:
import json

json.loads(text.strip())

{'Name': 'my-rule',
 'EventPattern': {'source': ['my.app'], 'detail-type': ['Order Placed']},
 'Targets': [{'Id': '1',
   'Arn': 'arn:aws:lambda:us-east-1:123456789012:function:ProcessOrder'}]}

In [13]:
from tracemalloc import stop

messages = []

add_user_message(messages, "Generate three different sample AWS CLI commands. Each should be very short")

add_assistant_message(messages, "Here are all three commands in a single block without any comments:\n```bash")

# Stop-sequence
text = chat(messages, stop_sequences=["```"])

text

'\naws s3 ls\n\naws ec2 describe-instances\n\naws iam list-users\n'

---

## **Prompt Evaluation**

This pipeline represents the foundation of most AI evaluation systems. While it may seem simple, you've just built the majority of what an eval pipeline actually does. The complexity comes in the details - better prompts, sophisticated grading, and performance optimizations.

In [14]:
import json

def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing a task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task"
    },
    ...additional
]
```
* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or
* Focus on tasks that do not require writing much code

Generate 3 objects
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [15]:
dataset = generate_dataset()

with open("dataset.json", "w") as f:
    json.dump(dataset, f, indent=2)

In [16]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

In [17]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)
    
    # TODO - Grading
    score = 10
    
    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

In [18]:
def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    
    return results

In [20]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [21]:
print(json.dumps(results, indent=2))

[
  {
    "output": "Here's a Python function that extracts the bucket name from an AWS S3 bucket ARN:\n\n```python\ndef extract_bucket_name_from_arn(arn):\n    \"\"\"\n    Extract the bucket name from an AWS S3 bucket ARN.\n    \n    Args:\n        arn (str): S3 bucket ARN in format 'arn:aws:s3:::bucket-name'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueError(\"ARN must be a string\")\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Validate ARN format\n    if len(parts) != 6:\n        raise ValueError(\"Invalid ARN format. Expected format: 'arn:aws:s3:::bucket-name'\")\n    \n    if parts[0] != 'arn':\n        raise ValueError(\"ARN must start with 'arn'\")\n    \n    if parts[1] != 'aws':\n        raise ValueError(\"ARN must specify 'aws' as partition\")\n    \n    if parts[2] != 's3':\n        raise ValueError(\"ARN

---

## **Model-Based Grading**

![graders](graders.png)

![evaluation](evaluation.png)